# Notebook 06: Text-to-Speech & Voice Synthesis

**Time:** 25 minutes  
**Prerequisites:** Notebook 05 complete  
**Goal:** Generate speech from text using modern TTS models (Kokoro, edge-tts)

This notebook will:
1. Use edge-tts for free, high-quality cloud TTS (no API key needed)
2. Try Kokoro TTS (82M params, highest quality score among open-source models)
3. Compare TTS options for voice agent applications
4. Explore voice selection and customization

> **Why this matters:** TTS is the output side of a voice agent. The 2025 TTS landscape has shifted dramatically -- Kokoro (82M params) achieves a 4.2 MOS score (highest among open-source), while CosyVoice2 leads in streaming latency (150ms). For a voice agent, you need fast, natural-sounding speech.

In [2]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

import src.audio_utils
importlib.reload(src.audio_utils)
from src.audio_utils import (
    synthesize_with_edge_tts,
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 06")

✓ Ollama client initialized
  Available models: ['nomic-embed-text:latest', 'qwen3.5:latest', 'gemma4:e2b', 'llama3:latest', 'granite4:latest']
  Default model: granite4:latest
Setup complete -- ready for Notebook 06


---

## Part 1: edge-tts (Free Cloud TTS)

**edge-tts** uses Microsoft's Edge browser read-aloud API -- completely free, no API key needed. It provides neural voices in 80+ languages.

```
pip install edge-tts
```

In [3]:
print("=" * 65)
print("Experiment 1: edge-tts Basic Synthesis")
print("=" * 65)
print()

test_text = "What I want to say is that the homework is pretty amazing. I do learned a lot from it."

edge_result = synthesize_with_edge_tts(
    text=test_text,
    voice="en-US-AriaNeural",
    save_path=os.path.join(outputs_dir, 'edge_demo.mp3'),
)

print(f"\nAudio saved. Play it with any media player.")

Experiment 1: edge-tts Basic Synthesis

EDGE TTS (voice: en-US-AriaNeural)
Text: What I want to say is that the homework is pretty amazing. I do learned a lot from it....
  Synthesized in 0.5s
  Saved to: ../outputs/edge_demo.mp3

Audio saved. Play it with any media player.


In [4]:
# Try playing audio in Jupyter
from IPython.display import Audio, display

audio_path = os.path.join(outputs_dir, 'edge_demo.mp3')
if os.path.exists(audio_path):
    display(Audio(audio_path))
else:
    print("Audio file not found.")

In [5]:
print("=" * 65)
print("Experiment 2: Voice Comparison")
print("=" * 65)
print()

voices = [
    ("en-US-AriaNeural", "Female, conversational"),
    ("en-US-GuyNeural", "Male, neutral"),
    ("en-GB-SoniaNeural", "Female, British"),
]

short_text = "Welcome to Week 3 of the ML Engineering course. Today we explore pretraining data pipelines."

for voice, desc in voices:
    print(f"\nVoice: {voice} ({desc})")
    result = synthesize_with_edge_tts(
        text=short_text,
        voice=voice,
        save_path=os.path.join(outputs_dir, f'voice_{voice.split("-")[2]}.mp3'),
    )

Experiment 2: Voice Comparison


Voice: en-US-AriaNeural (Female, conversational)
EDGE TTS (voice: en-US-AriaNeural)
Text: Welcome to Week 3 of the ML Engineering course. Today we explore pretraining data pipelines....
  Synthesized in 0.3s
  Saved to: ../outputs/voice_AriaNeural.mp3

Voice: en-US-GuyNeural (Male, neutral)
EDGE TTS (voice: en-US-GuyNeural)
Text: Welcome to Week 3 of the ML Engineering course. Today we explore pretraining data pipelines....
  Synthesized in 0.3s
  Saved to: ../outputs/voice_GuyNeural.mp3

Voice: en-GB-SoniaNeural (Female, British)
EDGE TTS (voice: en-GB-SoniaNeural)
Text: Welcome to Week 3 of the ML Engineering course. Today we explore pretraining data pipelines....
  Synthesized in 0.3s
  Saved to: ../outputs/voice_SoniaNeural.mp3


In [6]:
# TODO 1: Synthesize your own text
#
# Write a short paragraph about what you've learned this week,
# then synthesize it with your preferred voice.

print("=" * 65)
print("TODO 1: Custom TTS Synthesis")
print("=" * 65)
print()

my_text = "The home work is pretty amazing. One thing I want to say is that there are too many libraries need to be installed, it cause the environment messier."
my_voice = "en-US-AriaNeural"  # Change to your preferred voice

my_tts_result = synthesize_with_edge_tts(
    text=my_text,
    voice=my_voice,
    save_path=os.path.join(outputs_dir, 'my_tts_output.mp3'),
)

todo1_reflection = """
[YOUR REFLECTION HERE]
The sound is good, almost the same as human voice.
I like AriaNeural, it sounds more natural.
I think emotion is the limitation of cloud-based TTS, it may sound robotic and lack emotional expression.
- How natural did the synthesized speech sound?
- Which voice did you prefer and why?
- What are the limitations of cloud-based TTS like edge-tts?
"""

print()
print(todo1_reflection)

TODO 1: Custom TTS Synthesis

EDGE TTS (voice: en-US-AriaNeural)
Text: The home work is pretty amazing. One thing I want to say is that there are too many libraries need t...
  Synthesized in 0.4s
  Saved to: ../outputs/my_tts_output.mp3


[YOUR REFLECTION HERE]
The sound is good, almost the same as human voice.
I like AriaNeural, it sounds more natural.
I think emotion is the limitation of cloud-based TTS, it may sound robotic and lack emotional expression.
- How natural did the synthesized speech sound?
- Which voice did you prefer and why?
- What are the limitations of cloud-based TTS like edge-tts?



---

## Part 2: Kokoro TTS (Local, High-Quality)

**Kokoro** is a lightweight TTS model (82M parameters) that achieves the highest MOS score (4.2) among all open-source models. It runs on CPU and generates a 10-second clip in 0.3 seconds.

```
pip install kokoro soundfile
```

Available voices: `af_heart`, `am_adam`, `bf_emma`, `bm_george`, and more.

In [7]:
print("=" * 65)
print("Experiment 3: Kokoro TTS (Local)")
print("=" * 65)
print()

try:
    from src.audio_utils import synthesize_with_kokoro
    
    kokoro_result = synthesize_with_kokoro(
        text="The quality of your pretraining data defines the quality of your model. This is true for text, images, and audio alike.",
        voice="af_heart",
        speed=1.0,
        save_path=os.path.join(outputs_dir, 'kokoro_demo.wav'),
    )
    
    # Play in Jupyter
    kokoro_audio = os.path.join(outputs_dir, 'kokoro_demo.wav')
    if os.path.exists(kokoro_audio):
        display(Audio(kokoro_audio))
except ImportError as e:
    print(f"Kokoro not installed: {e}")
    print("Install with: pip install kokoro soundfile")
    print("\nKokoro is optional -- edge-tts works great as a free alternative.")
except Exception as e:
    print(f"Kokoro failed: {e}")

Experiment 3: Kokoro TTS (Local)

KOKORO TTS (voice: af_heart, speed: 1.0x)
Text: The quality of your pretraining data defines the quality of your model. This is true for text, image...


/home/conardd/miniconda3/envs/hk2/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/home/conardd/miniconda3/envs/hk2/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Generated 7.65s audio in 3.6s
  Saved to: ../outputs/kokoro_demo.wav


---

## Part 3: TTS Landscape (2025)

| Model | Size | MOS Score | Latency | Best For |
|-------|------|-----------|---------|----------|
| **Kokoro** | 82M | 4.2 (highest) | ~0.03 RTF | Quality + speed on CPU |
| **CosyVoice2** | 500M | ~4.0 | 150ms streaming | Real-time streaming |
| **Orpheus** | 150M-3B | ~3.9 | Variable | Emotion control, voice cloning |
| **Fish Speech** | 500M+ | ~4.1 | Variable | Multilingual, cloning |
| **edge-tts** | Cloud | ~4.0 | Network-dependent | Free, no setup |

In [8]:
# TODO 2: Discuss TTS for voice agents

print("=" * 65)
print("TODO 2: TTS Architecture for Voice Agents")
print("=" * 65)
print()

start = time.time()
response = client.generate(
    prompt="""I'm building a voice agent that needs to respond in real-time. Compare these TTS options:

1. **edge-tts** (free, cloud, ~200ms network latency)
2. **Kokoro** (82M params, local, ~30ms for 10s audio)
3. **CosyVoice2** (500M, streaming, 150ms first-byte latency)
4. **Orpheus** (150M-3B, emotion control, streaming)

For a voice agent that needs to:
- Respond within 500ms total (ASR + LLM + TTS)
- Run on a laptop (no GPU)
- Sound natural and conversational

Which would you recommend and why? What's the latency budget breakdown?""",
    system="You are a voice AI architect designing low-latency voice agents.",
    max_tokens=500,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo2_reflection = """
[YOUR REFLECTION HERE]
Kokoro sounds better, but it took long time to synthesize. 
If I use it in voice agent, the latency may be too high, it may cause bad user experience.
Edge-tts is good enough for applications.
- Which TTS model would you choose for your voice agent project?
- What's the biggest latency bottleneck in a voice pipeline (ASR vs LLM vs TTS)?
- How does edge-tts compare to local models like Kokoro for real-time use?
"""

print()
print(todo2_reflection)

TODO 2: TTS Architecture for Voice Agents

Response in 42.2s
Model: granite4:latest
Tokens: 182 in, 495 out
Stop reason: complete
Based on your requirements for real-time voice agent with low-latency, here are my recommendations:

**Recommendation**: Kokoro (edge-tts)

**Reasoning**:
1. Latency: With ~30ms latency per 10s audio segment and no network overhead (~200ms), Kokoro is the fastest option that can run locally on a laptop without GPU.

2. Quality: Despite being smaller than CosyVoice2, Kokoro's larger parameter count (82M) allows it to generate high-quality natural sounding speech for conversational interactions.

3. Compatibility: Running entirely offline eliminates network latency and any potential issues with cloud services or API limits.

4. Flexibility: Being a pure text-to-speech solution, it can be easily integrated with ASR and LLM components without additional complexity.

**Latency Budget Breakdown**:
- Text to Speech (TTS): 30ms per 10s audio segment
- Total TTS late

---

## Summary & Reflection

In [9]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'

full_reflection = f"""
### Part 1 - TTS Synthesis

{_todo1}

---

### Part 2 - Voice Agent TTS Architecture

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="06",
    section_title="Text-to-Speech & Voice Synthesis",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     1
Total input tokens:  182
Total output tokens: 495
Total cost:          $0.0080

Last 1 calls:
  1. [14:16:46] granite4:latest -- 182in/495out -- $0.0080


## Notebook 06 Complete!

**What you accomplished:**
- Synthesized speech with edge-tts (free, cloud-based)
- Explored Kokoro TTS (local, highest quality open-source)
- Compared TTS models for voice agent applications

**Key concepts:**
- edge-tts is the easiest entry point (free, no setup)
- Kokoro achieves highest MOS score (4.2) with only 82M params
- Latency budget matters: ASR (~100ms) + LLM (~200ms) + TTS (~100ms)

**Next:** Open **Notebook 07: Voice Agent with Pipecat**